# Spectroscopy workflow

This notebook walks through a practical spectroscopy flow for one qubit:
pick a readout power, inspect the resonator-qubit detuning with
`ckp_experiment()`, and then refine the readout and control frequencies.
Update `system_id`, `config_dir`, and `params_dir` for your setup before
running the cells.


## 1. Create an `Experiment`

Start with one qubit or one mux and cache the first qubit / resonator
labels for later cells.


In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    qubits=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

Q0 = exp.qubit_labels[0]
RQ0 = exp.resonator_labels[0]

print("target qubit:", Q0)
print("target resonator:", RQ0)

## 2. Connect and verify the readout path

`check_waveform()` is a quick sanity check before running longer sweeps.


In [ ]:
exp.connect()

waveform_result = exp.check_waveform(
    targets=Q0,
    n_shots=512,
    plot=True,
)

## 3. Sweep the readout amplitude

Use a small amplitude sweep to decide which readout drive level is worth
using for the finer spectroscopy steps.


In [ ]:
readout_amplitude_range = np.linspace(0.01, 0.08, 8)

amplitude_result = exp.sweep_readout_amplitude(
    targets=Q0,
    amplitude_range=readout_amplitude_range,
    initial_state="0",
    n_shots=512,
    plot=True,
)

## 4. Run a CKP experiment instead of reflection-coefficient fitting

`ckp_experiment()` performs the resonator / qubit detuning sweep for both
qubit states and returns fitted dispersive parameters in one step.


In [ ]:
chosen_readout_amplitude = 0.04

ckp_result = exp.ckp_experiment(
    target=Q0,
    resonator_detuning_range=np.linspace(-0.015, 0.015, 31),
    qubit_detuning_range=np.linspace(-0.08, 0.08, 41),
    resonator_drive_amplitude=chosen_readout_amplitude,
    plot=True,
    verbose=True,
    save_image=False,
)

print(
    {key: ckp_result[key] for key in ["f_r_0", "f_r_1", "f_r", "f_q", "chi", "kappa"]}
)

## 5. Refine the readout frequency

Once the readout amplitude is fixed, run the dedicated readout-frequency
calibration around the current operating point.


In [ ]:
readout_frequency_result = exp.calibrate_readout_frequency(
    targets=Q0,
    detuning_range=np.linspace(-0.010, 0.010, 41),
    readout_amplitudes={Q0: chosen_readout_amplitude},
    n_shots=512,
    plot=True,
)

## 6. Refine the control frequency

After the readout side is stable, calibrate the qubit drive frequency with
a short spectroscopy-style scan over detuning and pulse duration.


In [ ]:
control_frequency_result = exp.calibrate_control_frequency(
    targets=Q0,
    detuning_range=np.linspace(-0.08, 0.08, 41),
    time_range=np.arange(32, 257, 32),
    n_shots=512,
    plot=True,
)

## 7. Optionally persist the updated calibration note

Save and reload only when you want to make the new fit values the active
defaults for later experiments.


In [ ]:
# exp.calib_note.save()
# exp.reload()
